# Notebook 3 — Proces de Streaming în Timp Real

## Descriere generală

Acest notebook implementează un **sistem de monitorizare în timp real** a traficului aerian deasupra celor 5 aeroporturi europene monitorizate în proiect, folosind **Apache Kafka** și **Spark Structured Streaming**.

### Arhitectura sistemului

```
OpenSky Network API
        ↓  (get_states() la fiecare 60 secunde)
   Kafka Producer  →  Topic: 'zboruri'
        ↓
   Spark Structured Streaming
        ↓  (foreachBatch la fiecare 60 secunde)
   Procesare + Clasificare + Afișare raport
```

### Componentele sistemului

| Componentă | Tehnologie | Rol |
|---|---|---|
| **Producer** | Python + OpenSky API | Colectează pozițiile live ale avioanelor și le publică în Kafka |
| **Broker** | Apache Kafka | Sistemul de mesagerie care decuplează producătorul de consumator |
| **Consumer** | Spark Structured Streaming | Citește mesajele din Kafka, le procesează și generează rapoarte |


## 1. Inițializarea SparkSession cu suport Kafka

Pornim o sesiune Spark cu pachetul `spark-sql-kafka` care permite citirea din topicuri Kafka ca sursă de streaming. Configurăm și variabilele Hadoop necesare pe Windows pentru a evita erorile legate de permisiuni de fișiere.

In [2]:
import os
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] += r";C:\hadoop\bin"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("StreamingFlights") \
    .master("local[*]") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print("Sesiune pornită cu succes!")

Spark version: 3.5.3
Sesiune pornită cu succes!


## 2. Definirea schemei și a zonelor geografice monitorizate

### 2.1 Schema mesajelor Kafka

Kafka transmite mesajele ca bytes bruti. Definim schema JSON a mesajelor produse de OpenSky API, astfel încât Spark să știe cum să deserializeze fiecare câmp. Câmpurile provin direct din răspunsul `get_states()` al OpenSky Network:

| Câmp | Tip | Descriere |
|---|---|---|
| `icao24` | String | Adresa unică ICAO a transponderului |
| `callsign` | String | Indicativul zborului (ex: ROT384J) |
| `origin_country` | String | Țara de origine inferată din ICAO24 |
| `latitude`, `longitude` | Float | Coordonatele WGS-84 ale aeronavei |
| `baro_altitude` | Float | Altitudinea barometrică în metri |
| `velocity` | Float | Viteza față de sol în m/s |
| `vertical_rate` | Float | Rata de urcare/coborâre în m/s (pozitiv = urcă) |
| `on_ground` | Boolean | True dacă aeronava e pe pistă |
| `category` | Integer | Categoria aeronavei (0=necunoscut, 4=mare, 6=heavy) |

### 2.2 Zonele geografice ale aeroporturilor

Definim bounding box-urile în jurul fiecărui aeroport monitorizat. O aeronavă este considerată în zona unui aeroport dacă coordonatele sale se află în interiorul bounding box-ului corespunzător.

In [3]:
from pyspark.sql.functions import col, from_json, when, round as spark_round
from pyspark.sql.types import (
    StructType, StringType, FloatType,
    LongType, BooleanType, IntegerType
)
from datetime import datetime

# Schema mesajelor JSON primite din Kafka
schema = StructType() \
    .add("timestamp",      LongType()) \
    .add("icao24",         StringType()) \
    .add("callsign",       StringType()) \
    .add("origin_country", StringType()) \
    .add("latitude",       FloatType()) \
    .add("longitude",      FloatType()) \
    .add("baro_altitude",  FloatType()) \
    .add("velocity",       FloatType()) \
    .add("vertical_rate",  FloatType()) \
    .add("on_ground",      BooleanType()) \
    .add("category",       IntegerType())

# Bounding box-urile geografice ale aeroporturilor monitorizate
# (lat_min, lat_max) si (lon_min, lon_max)
AIRPORTS = {
    "EDDF": {"lat": (49.90, 50.20), "lon": (8.40,  8.80),  "name": "Frankfurt"},
    "EGLL": {"lat": (51.40, 51.55), "lon": (-0.55, -0.35), "name": "Heathrow"},
    "LFPG": {"lat": (48.95, 49.08), "lon": (2.45,  2.65),  "name": "Paris CDG"},
    "EHAM": {"lat": (52.25, 52.40), "lon": (4.70,  4.85),  "name": "Amsterdam"},
    "LROP": {"lat": (44.50, 44.65), "lon": (26.00, 26.20), "name": "Bucuresti"},
}

print(f"Schema definita cu {len(schema.fields)} campuri")
print(f"Aeroporturi monitorizate: {', '.join(AIRPORTS.keys())}")

Schema definita cu 11 campuri
Aeroporturi monitorizate: EDDF, EGLL, LFPG, EHAM, LROP


## 3. Citirea din Kafka și procesarea streamului

### 3.1 Conectarea la topicul Kafka

Spark Structured Streaming citește continuu mesajele din topicul `zboruri` de pe brokerul Kafka local. Fiecare mesaj conține poziția unui avion la un moment dat, serializată ca JSON.

In [4]:
df_kafka = (
    spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", "localhost:9092")
        .option("subscribe", "zboruri")
        .load()
)

print("Stream Kafka conectat la topicul 'zboruri'")
print(f"Schema raw Kafka: {df_kafka.schema.simpleString()}")

Stream Kafka conectat la topicul 'zboruri'
Schema raw Kafka: struct<key:binary,value:binary,topic:string,partition:int,offset:bigint,timestamp:timestamp,timestampType:int>


### 3.2 Parsarea JSON și clasificarea aeronavelor

Mesajele Kafka sosesc ca bytes bruti în coloana `value`. Aplicăm trei transformări:

1. **Deserializare JSON** — convertim bytes → string → structură cu `from_json()`
2. **Conversie viteză** — transformăm din m/s în km/h
3. **Clasificare status zbor** — pe baza `baro_altitude` și `on_ground` clasificăm fiecare aeronavă:
   - **LA SOL** — `on_ground = True`
   - **DECOLARE/ATERIZARE** — sub 1000m altitudine
   - **APROPIERE** — între 1000m și 3000m (faza finală de apropiere)
   - **SURVOLARE** — peste 3000m (zbor de croazieră sau tranzit)
4. **Clasificare direcție** — pe baza `vertical_rate` determinăm dacă aeronava urcă, coboară sau menține altitudinea

In [5]:

raw_df = (
    df_kafka
      .selectExpr("CAST(value AS STRING)")
      .select(from_json(col("value"), schema).alias("data"))
      .select("data.*")
      .withColumn("velocity_kmh", spark_round(col("velocity") * 3.6, 1))
      # Clasificare status zbor pe baza altitudinii si starii la sol
      .withColumn("flight_status",
          when(col("on_ground") == True,    "LA SOL")
         .when(col("baro_altitude") < 1000, "DECOLARE/ATERIZARE")
         .when(col("baro_altitude") < 3000, "APROPIERE")
         .otherwise(                        "SURVOLARE"))
      # Clasificare directie pe baza ratei verticale
      .withColumn("directie",
          when(col("vertical_rate") > 1,  "URCARE")
         .when(col("vertical_rate") < -1, "COBORARE")
         .otherwise(                      "NIVEL"))
)

print("Au fost aplicate o serie de transformari asupra streamului")

Au fost aplicate o serie de transformari asupra streamului


## 4. Funcția de procesare per batch (foreachBatch)

Spark Structured Streaming procesează datele în **micro-batch-uri** — la fiecare 30 de secunde colectează toate mesajele noi din Kafka și le trimite funcției `process_batch`.

Funcția primește un DataFrame Spark cu toate aeronavele detectate în intervalul respectiv și:
1. Îl convertește în Pandas pentru procesare rapidă
2. Filtrează aeronavele per aeroport folosind bounding box-urile geografice
3. Calculează statistici: total detectate, în aer vs la sol, viteză și altitudine medie
4. Afișează un raport structurat per aeroport și un sumar global

In [6]:
def process_batch(batch_df, epoch_id):
    """
    Primeste un DataFrame cu toate aeronavele detectate in intervalul respectiv.
    """
    df_p = batch_df.toPandas()
    ora = datetime.now().strftime('%H:%M:%S')

    print(f"\n{'='*70}")
    print(f"  MONITORIZARE TRAFIC AERIAN  |  Batch {epoch_id}  |  {ora}")
    print(f"{'='*70}")

    if df_p.empty:
        print("  Nicio aeronavă detectată.")
        print(f"{'='*70}")
        return

    total_detectate = 0

    # Procesam fiecare aeroport in parte
    for icao, info in AIRPORTS.items():
        zona = df_p[
            df_p['latitude'].between(*info["lat"]) &
            df_p['longitude'].between(*info["lon"])
        ]

        if zona.empty:
            print(f"\n  {info['name']} ({icao})")
            print(f"  {'-'*50}")
            print("  Nicio aeronavă detectată.")

        total_detectate += len(zona)
        in_aer  = zona[zona['on_ground'] == False]
        la_sol  = zona[zona['on_ground'] == True]
        aproape = zona[zona['baro_altitude'] < 3000]

        # Raport per aeroport
        print(f"\n  {info['name']} ({icao})")
        print(f"  {'-'*50}")
        print(f"  Total : {len(zona)} | In aer: {len(in_aer)} | La sol: {len(la_sol)}")

        if not in_aer.empty:
            print(f"  Viteza medie    : {in_aer['velocity_kmh'].mean():.1f} km/h")
            print(f"  Altitudine medie: {in_aer['baro_altitude'].mean():.0f} m")

        if not aproape.empty:
            print(f"  In apropiere/aterizare: {len(aproape)} aeronave")

        # Tabel cu aeronavele detectate
        print(f"\n  Aeronave detectate:")
        cols = ['callsign', 'origin_country', 'baro_altitude',
                'velocity_kmh', 'flight_status', 'directie']
        print(zona[cols].to_string(index=False))

        # Top tari de origine
        print(f"\n  Top tari origine:")
        for tara, cnt in zona['origin_country'].value_counts().head(5).items():
            print(f"    {tara:<20} {cnt}")

    # Sumar global batch
    print(f"\n{'─'*70}")
    print(f"  Total aeronave procesate in acest batch: {total_detectate}")
    print(f"{'='*70}")

## 5. Pornirea streaming-ului

Pornim query-ul de streaming cu următoarea configurație:
- **`outputMode("append")`** — fiecare batch adaugă noi date (nu suprascrie)
- **`foreachBatch(process_batch)`** — la fiecare batch se apelează funcția noastră de procesare
- **`trigger(processingTime='60 seconds')`** — Spark colectează și procesează mesajele la fiecare 60 de secunde
- **`awaitTermination()`** — păstrează notebook-ul activ până la oprirea manuală

In [ ]:
query = (
    raw_df.writeStream
          .outputMode("append")
          .foreachBatch(process_batch)
          .trigger(processingTime='60 seconds')
          .start()
)

print("Procesare la fiecare 60 secunde.")
query.awaitTermination()

Procesare la fiecare 60 secunde.

  MONITORIZARE TRAFIC AERIAN  |  Batch 0  |  17:50:58
  Nicio aeronavă detectată.

  MONITORIZARE TRAFIC AERIAN  |  Batch 1  |  17:52:02

  Frankfurt (EDDF)
  --------------------------------------------------
  Total : 23 | In aer: 10 | La sol: 13
  Viteza medie    : 303.1 km/h
  Altitudine medie: 642 m
  In apropiere/aterizare: 12 aeronave

  Aeronave detectate:
callsign origin_country  baro_altitude  velocity_kmh      flight_status directie
 PGT539T         Turkey            NaN          24.1             LA SOL    NIVEL
   DETCY        Germany      30.480000         115.2 DECOLARE/ATERIZARE COBORARE
  DLH543        Germany    2377.439941         566.4          APROPIERE COBORARE
 RAM810D        Morocco            NaN          38.9             LA SOL    NIVEL
  TKJ2ER         Turkey      15.240000          70.4             LA SOL    NIVEL
  DLH293          Italy            NaN           0.0             LA SOL    NIVEL
   DIGAH        Germany     320.

## 6. Oprirea streaming-ului


In [ ]:
# Opreste toate stream-urile active
for q in spark.streams.active:
    print(f"Opresc query-ul: {q.name}")
    q.stop()

print("Toate stream-urile au fost oprite.")
# spark.stop()

## 7. Statistici despre topicul Kafka

Citim topicul Kafka în mod batch (nu streaming) pentru a vedea câte mesaje totale au fost publicate de producer de la pornirea sistemului. Aceasta este o metodă utilă pentru a verifica că producer-ul funcționează corect și că mesajele ajung în Kafka.

In [ ]:
df_count = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "zboruri") \
    .option("startingOffsets", "earliest") \
    .load()

total = df_count.count()
print(f"Total mesaje in topicul 'zboruri': {total}")